# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*\

A page is flagged for review if it's ranking in a position where clicks are known to be weak (position worse than 10, where CTR has already dropped below 0.35 in my bucket check) and it's actively trending worse, not just sitting still. I'm deliberately excluding the noisy tail bucket (position > 100, n=1,991) from triggering the strongest reason code, since that bucket's numbers are driven by near-zero-impression pages, not real signal  those get downgraded to a weaker/monitor-only code instead.

The idea: don't flag every low-ranking page flag the ones that are both underperforming and getting worse, since those are the ones actively losing ground and worth someone's attention now, versus pages that are just chronically low but stable (lower priority) or too sparse in data to trust (exclude/monitor only).

In [1]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)


In [2]:
from datasets import load_dataset
cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']
ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)


df = ds.select_columns(cols_needed).to_pandas()
cols_needed = [ 'content_hash_id', 'word_count', 'is_published']
ds_content = load_dataset("FlyRank/internship-warehouse",
    data_files="dim_content.parquet", split="train", token=hf_token)
df_content = ds_content.select_columns(cols_needed).to_pandas()

df  = df.merge(df_content, on='content_hash_id', how='left', validate="m:1")
print(df.shape)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 8)


In [3]:
import numpy as np
df = df[(df['is_published'] == True)].copy()

df.loc[df['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan

df = df[(df['gsc_avg_position'].isna()) | (df['gsc_avg_position'] <= 100)]


In [4]:
tail_bucket = df[
    (df['gsc_avg_position'] > 100) & (df['gsc_avg_position'] <= 500)
]
print(tail_bucket[['gsc_impressions', 'gsc_clicks']].describe())

       gsc_impressions  gsc_clicks
count              0.0         0.0
mean               NaN         NaN
std                NaN         NaN
min                NaN         NaN
25%                NaN         NaN
50%                NaN         NaN
75%                NaN         NaN
max                NaN         NaN


In [5]:
import pandas as pd
df['pos_bucket'] = pd.cut(df['gsc_avg_position'], bins=[0,3,10,20,100,500])
df['CTR'] = (df['gsc_clicks'] / df['gsc_impressions']) * 100

bucket_table = df.groupby('pos_bucket')['CTR'].agg(['mean', 'count'])
print(bucket_table)

/tmp/ipykernel_2345/2597075450.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table = df.groupby('pos_bucket')['CTR'].agg(['mean', 'count'])


                mean    count
pos_bucket                   
(0, 3]      0.491338   563904
(3, 10]     0.347488  1454694
(10, 20]    0.276992   519100
(20, 100]   0.127814   906281
(100, 500]       NaN        0


In [7]:
df['report_date'] = pd.to_datetime(df['report_date'])
df.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)

pos_diff = df.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df.groupby('content_hash_id')['report_date'].diff().dt.days

df['trend_direction'] = (pos_diff / days_diff).fillna(0)

In [8]:
df.groupby('pos_bucket')['trend_direction'].agg(['mean', 'count'])

/tmp/ipykernel_2345/1832354741.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('pos_bucket')['trend_direction'].agg(['mean', 'count'])


,mean,count
pos_bucket,,
"(0, 3]",-2.473924,563904
"(3, 10]",-1.540452,1454694
"(10, 20]",-0.609259,519100
"(20, 100]",4.571745,906281
"(100, 500]",NaN,0


**CTR vs Position behind the CTR-fix logic**
- CONFIRMED: CTR declines cleanly and monotonically from 0.491 (position 0-3) to 0.128 (position 20-100) across all rows, confirming the CTR-fix logic pattern. The (100, 500] tail bucket is empty (n=0) here since my position <= 100 filter excludes it entirely as part of the position==0 cleanup I did this pass.

**Trend_direction vs position bucket (staleness-adjacent)**
- CONFIRMED: I bucketed by gsc_avg_position and computed mean trend_direction per bucket (negative = improving position, positive = worsening). Clean pattern across all rows: top-ranked pages (0-10) trend improving (-2.47 to -1.54), mid-ranked pages (10-20) are declining slightly (-0.61), and poorly-ranked pages (20-100) are actively declining further (+4.57). The (100, 500] bucket is empty (n=0) here for the same filtering reason above.

Verdict: CONFIRMED

In [9]:
!git clone https://github.com/ocedev112/flyrank_oce_dev
%cd flyrank_oce_dev/

Cloning into 'flyrank_oce_dev'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 213 (delta 105), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (213/213), 3.20 MiB | 15.85 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/flyrank_oce_dev


In [10]:
import os
print(os.getcwd())

/content/flyrank_oce_dev


In [11]:
import os
os.makedirs('work/outputs', exist_ok=True)
print(os.path.exists('work/outputs'))

True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

os.makedirs('work/outputs', exist_ok=True)

#Encoded Rule
position = df['gsc_avg_position'].values
trend = df['trend_direction'].values
impressions = df['gsc_impressions'].values
clicks = df['gsc_clicks'].values

nan_mask = pd.isna(position)

impressions_safe = np.where(impressions <= 0, 1, impressions)
clicks_safe = np.clip(clicks, 0, impressions_safe)
ci_low, ci_high = proportion_confint(clicks_safe, impressions_safe, alpha=0.10, method='wilson')
interval_width = ci_high - ci_low

low_impressions_mask = (~nan_mask) & (interval_width > 0.15)
low_conf_mask = (~nan_mask) & (position > 100)
healthy_mask = (~nan_mask) & (position <= 10) & (~low_impressions_mask)
declining_mask = (~nan_mask) & (position > 10) & (position <= 100) & (trend > 0) & (~low_impressions_mask)

position_weakness = np.clip((position - 10) / 90, 0, 1)
trend_worsening = np.clip(trend / 100, 0, 1)
score = np.round(position_weakness * 60 + trend_worsening * 40, 4)
score[nan_mask | low_conf_mask | healthy_mask | low_impressions_mask] = 0.0

df['action_score'] = score

#Reason Code
reason_code = np.full(len(df), 'WEAK_BUT_STABLE', dtype=object)
action = np.full(len(df), 'monitor', dtype=object)

reason_code[nan_mask] = 'NO_DATA'
action[nan_mask] = 'monitor'

reason_code[low_impressions_mask] = 'LOW_CONFIDENCE_SIGNAL'
action[low_impressions_mask] = 'monitor'

reason_code[low_conf_mask] = 'LOW_CONFIDENCE_SIGNAL'
action[low_conf_mask] = 'monitor'

reason_code[healthy_mask] = 'HEALTHY'
action[healthy_mask] = 'no_action'

reason_code[declining_mask] = 'DECLINING_UNDERPERFORMER'
action[declining_mask] = 'review_priority'

df['reason_code'] = reason_code
df['action'] = action

ranked = df.sort_values('action_score', ascending=False)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(ranked['reason_code'].value_counts())
print(ranked.head(10)[['content_hash_id','gsc_avg_position','trend_direction','gsc_impressions', 'action_score','reason_code','action']])

del position, trend, impressions, clicks, impressions_safe, clicks_safe, ci_low, ci_high, interval_width, reason_code, action, position_weakness, trend_worsening, score, nan_mask, low_conf_mask, healthy_mask, declining_mask, ranked
import gc
gc.collect()

reason_code
NO_DATA                     6086748
LOW_CONFIDENCE_SIGNAL       1652615
HEALTHY                     1151965
DECLINING_UNDERPERFORMER     361625
WEAK_BUT_STABLE              277774
Name: count, dtype: int64
                  content_hash_id  gsc_avg_position  trend_direction  \
7648470  content_a3728f3736d7b3c4         98.882353        98.215686   
8583588  content_4c8f60c8875f8cc6         99.345313        89.507889   
8584956  content_450304ab9cd8fcbe         95.785714        95.341270   
7648313  content_1b46fa0fc079e148         96.863636        92.017483   
8585314  content_47be001e51cc63ea         95.240741        92.407407   
8585157  content_262ab3e5c2d0a4ea         98.814815        85.235504   
8583517  content_b767c33cc4eff413         96.959799        87.969749   
8584719  content_f357db8f98610538         98.590909        82.816716   
8584716  content_250c4a93d3c5f9b9         97.423077        82.923077   
7650961  content_d7db42380f907ad9         95.000000        86.

0

In [13]:
import json

ranked = pd.read_csv('work/outputs/baseline_action_score.csv')

metrics = {
    "run_date": "2026-03",
    "n_rows_total": len(ranked),
    "reason_code_counts": ranked['reason_code'].value_counts().to_dict(),
    "signal_checks": {
        "ctr_vs_position": {
            "verdict": "CONFIRMED",
        },
        "trend_vs_position": {
            "verdict": "CONFIRMED",
        }
    },
    "top10_content_hash_ids": ranked.head(10)['content_hash_id'].tolist()
}

with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print(json.dumps(metrics, indent=2, default=str))

{
  "run_date": "2026-03",
  "n_rows_total": 9530727,
  "reason_code_counts": {
    "NO_DATA": 6086748,
    "LOW_CONFIDENCE_SIGNAL": 1652615,
    "HEALTHY": 1151965,
    "DECLINING_UNDERPERFORMER": 361625,
    "WEAK_BUT_STABLE": 277774
  },
  "signal_checks": {
    "ctr_vs_position": {
      "verdict": "CONFIRMED"
    },
    "trend_vs_position": {
      "verdict": "CONFIRMED"
    }
  },
  "top10_content_hash_ids": [
    "content_a3728f3736d7b3c4",
    "content_4c8f60c8875f8cc6",
    "content_450304ab9cd8fcbe",
    "content_1b46fa0fc079e148",
    "content_47be001e51cc63ea",
    "content_262ab3e5c2d0a4ea",
    "content_b767c33cc4eff413",
    "content_f357db8f98610538",
    "content_250c4a93d3c5f9b9",
    "content_d7db42380f907ad9"
  ]
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

- Row 0 review_priority: DECLINING_UNDERPERFORMER: position 98.9, trend +98.2, 17 impressions, action_score 98.54. Near total ranking collapse. This row already passed the Wilson interval check, so the old worry about "is 17 impressions enough to trust" is now baked into inclusion rather than something to flag separately. What would make this wrong: I'd still want to check content_created_date in case this is a young page still settling rather than a page that was doing well and is now falling.

- Row 1 review_priority: DECLINING_UNDERPERFORMER: position 99.3, trend +89.5, 640 impressions, action_score 95.37. Real traffic volume, and a decline at this scale is unlikely to be sampling noise. This is still my most credible pick for the same reason as before. What would make this wrong: worth checking content_updated_date or last_optimized_date to see if a recent edit lines up with the drop, correlation rather than proof but a reasonable next check.

- Row 2 review_priority: DECLINING_UNDERPERFORMER: position 95.8, trend +95.3, 28 impressions, action_score 95.33. A steep decline with a moderate sample size behind it. What would make this wrong: I'd check whether this content_hash_id belongs to a content_type like a category or hub page, where a position in the 90s can be structurally normal rather than a real regression.

- Row 3 review_priority: DECLINING_UNDERPERFORMER: position 96.9, trend +92.0, 22 impressions, action_score 94.72. Consistent with the other high-decline rows. What would make this wrong: worth a keyword_hash_id check to see if this page is being cannibalized by a newer page targeting the same query rather than genuinely losing relevance.

- Row 4 review_priority: DECLINING_UNDERPERFORMER: position 95.2, trend +92.4, 54 impressions, action_score 93.79. This row did not make the original top 10 under the impressions less than 10 rule, it was previously scored as WEAK_BUT_STABLE or filtered out by the old cutoff logic even though the underlying decline was real. This is the clearest example of the old rule hiding a legitimate signal because it only checked a raw impression count, not whether the CTR estimate itself was actually unstable.

- Row 5 review_priority: DECLINING_UNDERPERFORMER: position 98.8, trend +85.2, 270 impressions, action_score 93.30. Strong impression volume, so this decline reads as trustworthy. What would make this wrong: at this position, worth checking whether the page recently lost a featured snippet or similar SERP feature, since that can produce a large apparent position swing that isn't about content quality at all.

- Row 6 review_priority: DECLINING_UNDERPERFORMER: position 97.0, trend +88.0, 398 impressions, action_score 93.16. One of the highest impression counts in the top 10, this is a well supported decline. What would make this wrong: unlikely to be noise given the volume, more useful to check whether a competitor recently outranked this page for the same query set.

- Row 7 review_priority: DECLINING_UNDERPERFORMER: position 98.6, trend +82.8, 88 impressions, action_score 92.19. A reasonably sized sample backing a clear decline. What would make this wrong: worth checking whether this page's word_count or last_optimized_date shows it was recently touched, since an edit that coincided with the decline would point at cause rather than just correlation.

- Row 8 review_priority: DECLINING_UNDERPERFORMER: position 97.4, trend +82.9, 26 impressions, action_score 91.45. What would make this wrong: same content_type caveat as row 2, worth confirming this isn't a page where a position in the high 90s is expected rather than a regression.

- Row 9 review_priority: DECLINING_UNDERPERFORMER: position 95.0, trend +86.0, 17 impressions, action_score 91.07. What would make this wrong: same young page caveat as row 0, worth checking content_created_date before treating this as an established page in decline.

The main shift from the original review is that the borderline cases sitting right at the old impressions equals 10 cutoff, rows 5 and 9 in the original list, are gone from the top 10 entirely. In their place are rows like the new row 4, at 54 impressions, that the old rule either missed or under scored because it only checked a raw impression count rather than whether the click through rate estimate was actually stable. That is the change the Wilson interval was meant to produce, and this top 10 is the first direct evidence it is doing that rather than just moving the same rows around.

In [14]:
top10 = pd.read_csv('work/outputs/baseline_action_score.csv', nrows=10)
print(top10[['content_hash_id', 'gsc_avg_position', 'trend_direction', 'gsc_impressions', 'gsc_clicks', 'CTR', 'action_score', 'reason_code', 'action']].to_string())review_priority$0

            content_hash_id  gsc_avg_position  trend_direction  gsc_impressions  gsc_clicks    CTR  action_score               reason_code           action
0  content_a3728f3736d7b3c4         98.882353        98.215686               17           0  0.000       98.5412  DECLINING_UNDERPERFORMER  review_priority
1  content_4c8f60c8875f8cc6         99.345313        89.507889              640           4  0.625       95.3667  DECLINING_UNDERPERFORMER  review_priority
2  content_450304ab9cd8fcbe         95.785714        95.341270               28           0  0.000       95.3270  DECLINING_UNDERPERFORMER  review_priority
3  content_1b46fa0fc079e148         96.863636        92.017483               22           0  0.000       94.7161  DECLINING_UNDERPERFORMER  review_priority
4  content_47be001e51cc63ea         95.240741        92.407407               54           0  0.000       93.7901  DECLINING_UNDERPERFORMER  review_priority
5  content_262ab3e5c2d0a4ea         98.814815        85.235504  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

7 of my top 10 picks still show 0 clicks despite the Wilson interval change, so a low click count alone isn't disqualifying anymore, what matters now is whether the confidence interval on that click rate is tight enough to trust. Impressions across the top 10 range from 17 to 640, all above the old raw cutoff of 10, so the interval-based filter isn't pulling in thinner rows than before, it's doing something different: it let row 4 in at 54 impressions, a row the earlier raw-count version scored as WEAK_BUT_STABLE or excluded outright even though the decline was real and the impression count wasn't actually that thin.

Row 1 is still the one pick I trust most without reservation. 640 impressions and 4 clicks is a real sample, and a decline at that volume is unlikely to be noise.

The two borderline picks from before, sitting right at the old impressions equals 10 line, are gone from this top 10 entirely. That's the change I was hoping to see. The rule is no longer including or excluding rows based on whether they happened to clear an arbitrary count, it's including them based on whether the CTR estimate itself is stable enough to act on. I'd still want to sanity check the interval width threshold I used, 0.15, the same way I flagged 10 as unjustified last time. I haven't swept that number yet, so I'm not fully confident it's the right cutoff either, just that it's a better kind of cutoff than a raw count.

Leakage check

No future windows leaked in. All features come from report_date within March 2026. trend_direction is a diff computed within each content_hash_id, ordered by report_date, so it only uses information up to that day. I did not touch the sample table, which covers June 2026.

No outcome flags leaked in. The rule uses gsc_avg_position, gsc_clicks, gsc_impressions, and word_count. I left out last_optimized_date and optimization_eligible_date on purpose, since those reflect whether action was already taken, not something I would know before flagging a page. The Wilson interval is built from gsc_clicks and gsc_impressions on the same row, both already in scope, so it doesn't introduce any feature that wasn't already part of the rule.

One thing worth flagging: content_updated_date is not part of the rule, but I mentioned it a few times in the top 10 notes as something worth checking manually. If its timestamp does not line up cleanly with report_date

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.